In [2]:
import sys
import pandas as pd
from pathlib import Path

from panel_utils import (clean_panel_df,
                         mean_WTD_col_for_unit,
                         create_categorical_cols_in_df,
                         create_FE_columns_in_df,
                         pyfixest_fit_FE,
                         save_panel_model_results, 
                         compute_climate_trend_IWU_fe_model)

%load_ext autoreload
%autoreload 2

In [3]:
PROJECT_ROOT = Path().resolve().parent.parent

sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend')

## Fixed-effects specification

The panel regression takes the form:

    IWU_v1_mm ~ Precip_mm:HUC8 + Tmean_C:HUC8 + Monthly_Q_mm:HUC8 + i(HUC8, year)
                | HUC8_month_fe + year_fe

estimated with Driscoll-Kraay standard errors over a 24-month bandwidth to
handle serial and cross-sectional residual correlation.

### What each absorber does

**HUC8 × month FE.** Absorbs each watershed's seasonal climatology — the
time-invariant mean of consumptive use in every (HUC8, month) cell. Because
HUC8 is nested within state, this term also absorbs time-invariant state-level
baselines (e.g. mean differences between Kansas and Nevada watersheds).

**HUC8-specific linear time trend** (`i(HUC8, year)`). Absorbs smooth,
monotonic long-run drift at the watershed scale: changes in irrigated area,
cropping mix, irrigation-efficiency adoption, and pumping technology. These
factors are watershed-specific, time-varying but non-seasonal, and are not
captured by either FE absorber.

**Year FE.** Absorbs *nationally common* shocks: federal farm-bill changes,
national commodity-price swings, and any climate or policy events that moved
all panel units in the same direction in a given year.

### Choice of year FE rather than state × year FE

A state × year FE is the more common choice in the recent panel-water-use
literature (Blumberg & Warziniack 2026; Davenport et al. 2021). In our
setting it absorbs not only state-specific policy and pricing shocks but
also the state-wide annual climate signal — the very quantity we wish to
attribute irrigation variability to. Because western-US hydroclimate is
spatially heterogeneous (ENSO, PDO, and regional drought events affect
different states in opposing directions), a state × year FE strips the
dominant identifying variation for the climate coefficients.

Year FE instead absorbs only the across-all-HUC8s common component of annual
variation. Spatially heterogeneous climate variation — the bulk of the
western-US wet/dry signal — survives as identifying variation for β. The
HUC8-specific linear trend separately handles slow drift that a state-time
FE might otherwise be expected to control.

### Robustness

We compared two specifications:

  (i)  HUC8 × month  +  state × year  FE
  (ii) HUC8 × month  +  year          FE   [adopted]

HUC8-level climate coefficients differed by less than 1% between the two,
indicating that state-by-year confounders beyond what year FE absorbs do
not materially bias β in this panel. The state × year specification yields
a substantively identical β but produces region-level IWU_climate aggregates
that are non-interpretable against state-scale hydroclimate context, because
the state × year FE has already absorbed precisely the signal that the
regional plots are designed to display. For example, if we use state × year FE,
the IWU response to wet vs dry year completely shifts, generating counterintuitive 
results.

### Identification scope

β_P and β_T identify the marginal response of consumptive irrigation water
use to within-HUC8 monthly climate variation, after seasonality, slow
watershed drift, and nationally common annual shocks are removed. The
resulting IWU_climate decomposition (β_P · Precip_anomaly + β_T · Tmean_anomaly)
is interpretable as the climate-attributable component of interannual
irrigation variability at HUC8 and regional scales.

# Loading and preparing panel df

In [4]:
pdf = pd.read_csv(PROJECT_ROOT / 'Data_main/panel_data/panel_data_monthly.csv')
print(pdf.columns, '\n')
print(f'Monthly df length: {len(pdf)}', '\n')
print(f'Number of HUC8: {pdf["HUC8"].nunique()}', '\n')
print(f'Number of Irrigated HUC8: {pdf[pdf["Irrigated"] == True]["HUC8"].nunique()}', '\n')
print(f'Number of unique aquifer_region: {pdf["AQ_Region"].nunique()}', '\n')
print(f'Unique aquifer_region: {pdf["AQ_Region"].unique()}', '\n')

# clean the panel df
# removing rows with NaN and selecting Irrigated HUC8s
pdf = clean_panel_df(pdf, 
                     nan_cols_to_consider=['Precip_mm', 'Tmean_C', 'Monthly_Q_mm'],
                     save_path=PROJECT_ROOT / 'Data_main/panel_data/panel_data_monthly_cleaned.csv',
                     years_to_consider=list(range(1986, 2020)))

# averaging WTD columns for each aquifer_region
pdf = mean_WTD_col_for_unit(pdf, 'WTD_Rnd_Frst_m', unit_col='AQ_Region')

# categorical column creation
categorical_config = {
    'HUC8': {
        'col_name' : 'HUC8',
        'assigned_categories' : list(pdf.HUC8.unique()),
        'impose_order'    : False
    },
    
    'Water_source': {
        'col_name' : 'Water_source',
        'assigned_categories' : ['Surface Water', 'Groundwater', 'Conjunctive'],
        'impose_order'    : False
    }
}

pdf = create_categorical_cols_in_df(pdf, categorical_config)

print(pdf.info())

print('Total HUC8 units in the final panel dataframe (after cleaning):', pdf.HUC8.nunique())


Index(['Winter_Precip_mm', 'Irr_area_ha', 'ET_mm', 'IWU_v1_mm', 'IWU_v2_mm',
       'IWU_v3_mm', 'IWU_ACFT', 'Precip_mm', 'Tmean_C', 'WTD_Rnd_Frst_m',
       'WTD_USGS_m', 'HUC8', 'State', 'AQ_NAME', 'AQ_Region', 'year', 'month',
       'Monthly_Q_mm', 'Irrigated', 'Water_source'],
      dtype='object') 

Monthly df length: 315742 

Number of HUC8: 1187 

Number of Irrigated HUC8: 581 

Number of unique aquifer_region: 28 

Unique aquifer_region: [nan 'RG_NM' 'BR_AZ_East' 'UCRB_CO' 'HPA_CO' 'RG_CO' 'HPA_TX_North'
 'BR_NV_Central' 'HPA_OK' 'HPA_KS_West' 'HPA_KS_East' 'UCRB_UT'
 'BR_AZ_West' 'HPA_TX_South' 'CV_CA_Tulare' 'CV_CA_SanJoaquin' 'DBA_CO'
 'BR_UT_South' 'HPA_NE' 'BR_NV_North' 'BR_NV_West' 'CV_CA_Sacramento'
 'UCRB_WY' 'BR_UT_North' 'SRP_ID_West' 'Will_OR' 'SRP_ID_East' 'CP_WA'
 'CP_OR'] 



INFO:panel_utils:Cleaned dataframe saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/panel_data/panel_data_monthly_cleaned.csv  |  shape: (134820, 20)




Step 1: Cleaned panel dataframe by dropping NaN rows and selecting irrigated HUC8s.
-------------------------------------------------------------------------------- 

HUC8s before cleaning the dataframe: 1187
HUC8s after cleaning the dataframe: 580

STEP 2: WTD column "WTD_Rnd_Frst_m" replaced with unit-level mean → "WTD_mean_m"
-------------------------------------------------------------------------------- 


STEP 3: Categorical columns created: ['HUC8', 'Water_source']
-------------------------------------------------------------------------------- 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134820 entries, 0 to 134819
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   Winter_Precip_mm  134820 non-null  float64 
 1   Irr_area_ha       134820 non-null  float64 
 2   ET_mm             134762 non-null  float64 
 3   IWU_v1_mm         132266 non-null  float64 
 4   IWU_v2_mm         133172 non-

# Model 2

`IWU_v1_mm ~ (HUC8, year) trend + Precip_anomaly:HUC8 + Tmean_anomaly:HUC8 + Monthly_Q_mm:HUC8 + | HUC8_month_fe + year_fe`

In [8]:
# Fixed-effects column creation
FE_config = {
            'HUC8_month' : ['HUC8', 'month'],
            'year' : ['year']
        }

pdf = create_FE_columns_in_df(pdf, FE_config,
                              year_col='year', 
                              month_col='month')


# FE regression model
fe_mod_2 = pyfixest_fit_FE(df=pdf, target_col='IWU_v1_mm', 
                           regressor_cols=['Precip_mm', 'Tmean_C', 
                                           'Monthly_Q_mm'], 
                           fe_cols=['HUC8_month_fe',
                                    'year_fe'],
                           include_base_regressors=False, 
                           interaction_dict={'Precip_mm': 'HUC8',
                                             'Tmean_C': 'HUC8',
                                             'Monthly_Q_mm': 'HUC8'},
                           add_linear_trend=True,
                           unit_col='HUC8', trend_col='year',
                           vcov_method='DK',
                           vcov_col='time_id', 
                           bandwidth=24,
                           save_pyfixest_model=True,
                           model_save_path=PROJECT_ROOT / 'Data_main/Results/models/fe_mod_2.pkl')

fe_mod_2.summary()


STEP 4: Fixed effects columns created for: ['HUC8_month', 'year']
***** Not all FE columns will be included in the regression. *****
-------------------------------------------------------------------------------- 



/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            3 variables dropped due to multicollinearity.
            The following variables are dropped: ['C(HUC8)[12100401]:year', 'Monthly_Q_mm:HUC8[16040107]', 'Monthly_Q_mm:HUC8[16040108]'].
            
  warnings.warn(
INFO:panel_utils:Pyfixest model fitted. Formula: IWU_v1_mm ~ i(HUC8, year) + Precip_mm:HUC8 + Tmean_C:HUC8 + Monthly_Q_mm:HUC8 | HUC8_month_fe + year_fe | vcov: DK
INFO:panel_utils:Pyfixest model saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/models/fe_mod_2.pkl


###

Estimation:  OLS
Dep. var.: IWU_v1_mm, Fixed effects: HUC8_month_fe+year_fe
Inference:  DK
Observations:  132266

| Coefficient                 |   Estimate |   Std. Error |   t value |   Pr(>|t|) |       2.5% |      97.5% |
|:----------------------------|-----------:|-------------:|----------:|-----------:|-----------:|-----------:|
| C(HUC8)[11080002]:year      |     -0.292 |        0.263 |    -1.110 |      0.268 |     -0.809 |      0.226 |
| C(HUC8)[11080004]:year      |     -0.361 |        0.318 |    -1.136 |      0.257 |     -0.988 |      0.266 |
| C(HUC8)[11080008]:year      |     -0.612 |        0.340 |    -1.802 |      0.073 |     -1.281 |      0.057 |
| C(HUC8)[13020101]:year      |     -0.201 |        0.255 |    -0.789 |      0.431 |     -0.702 |      0.301 |
| C(HUC8)[13020102]:year      |     -0.271 |        0.243 |    -1.112 |      0.267 |     -0.750 |      0.209 |
| C(HUC8)[13020203]:year      |     -0.277 |        0.229 |    -1.209 |      0.228 |     -0.727 |      0

`Notes on coef`
- Almost all P and T coefficients are statistically significant.
- A majority portion of Q coefficients are statistically insignificant, which can be becuase of its being naturalized Q. We couldn't incorporate actual Q due to endogeneity concern.

In [10]:
df_fe_mod = save_panel_model_results(
        model=fe_mod_2,
        model_name='model_2',
        incorporate_water_source_info=True,
        panel_df=pdf,
        output_dir=PROJECT_ROOT / 'Data_main/Results',
        shapefile=PROJECT_ROOT / 'Data_main/ref_shapes/WestUS_HUC8_processed.shp',
        spatial_unit_col='HUC8',
        rename_sp_unit='HUC8',
        shp_join_col='HUC8',
        save_csv=True,
        save_shapefile=True)

INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_2_results.csv
/Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Codes/panel_reg/panel_utils.py:1365: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  spatial_gdf.to_file(shp_path)
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'trend_97.5%' to 'trend_97.5'
  ogr_write(
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'trend_Estimate' to 'trend_Esti'
  ogr_write(
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'trend_p-value' to 'trend_p-va'
  ogr_write(
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyo

In [11]:
# loading panel model data and trained model results
cleaned_pdf = pd.read_csv(PROJECT_ROOT / 'Data_main/panel_data/panel_data_monthly_cleaned.csv')
results_csv = PROJECT_ROOT / 'Data_main/Results/model_2_results.csv'
output_dir = PROJECT_ROOT / 'Data_main/Results'

compute_climate_trend_IWU_fe_model(
        trained_fe_model_pkl=PROJECT_ROOT / 'Data_main/Results/models/fe_mod_2.pkl',
        panel_df=cleaned_pdf,
        predictors_list=['Precip_mm', 'Tmean_C', 'Monthly_Q_mm'],
        coef_csv=results_csv,
        output_dir=output_dir,
        model_name='model_2')

INFO:panel_utils:Monthly predicted IWU saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_2_predicted_IWU_monthly.csv
INFO:panel_utils:Annual  predicted IWU saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_2_predicted_IWU_annual.csv


(            HUC8       State AQ_Region   Water_source  year  month  Irrigated  \
 0       11080002  New Mexico       NaN            NaN  1986      4       True   
 1       11080004  New Mexico       NaN            NaN  1986      4       True   
 2       11080008  New Mexico       NaN            NaN  1986      4       True   
 3       13020101  New Mexico     RG_NM  Surface Water  1986      4       True   
 4       13020102  New Mexico     RG_NM  Surface Water  1986      4       True   
 ...          ...         ...       ...            ...   ...    ...        ...   
 134815  10230001    Nebraska       NaN            NaN  2019     10       True   
 134816  10230006    Nebraska       NaN            NaN  2019     10       True   
 134817  10270102      Kansas       NaN            NaN  2019     10       True   
 134818  10270104      Kansas       NaN            NaN  2019     10       True   
 134819  10240007      Kansas       NaN            NaN  2019     10       True   
 
         Preci

## Model 3

`IWU_v1_mm ~ Precip_anomaly:Water_source_cat + Tmean_anomaly:Water_source_cat + MMonthly_Q_mm:Water_source_cat | HUC8_month_fe + year_fe`

Same fixed-effects structure as Model 2, but interactions are with **water source type** (SW-dependent=0 / Conjunctive=1 / GW-dependent=2) rather than HUC8. Estimates a single sensitivity coefficient per water-source group.

In [13]:
fe_mod_3 = pyfixest_fit_FE(
    df=pdf,
    target_col='IWU_v1_mm',
    regressor_cols=['Precip_mm', 'Tmean_C', 'Monthly_Q_mm'],
    fe_cols=['HUC8_month_fe', 
            'year_fe'],
    include_base_regressors=False,
    interaction_dict={
        'Precip_mm'  : 'Water_source',
        'Tmean_C'   : 'Water_source',
        'Monthly_Q_mm'    : 'Water_source',
    },
    add_linear_trend=False,
    unit_col='HUC8', trend_col='year',
    vcov_method='DK',
    vcov_col='time_id',
    bandwidth=24
)

fe_mod_3.summary()

INFO:panel_utils:Pyfixest model fitted. Formula: IWU_v1_mm ~ Precip_mm:Water_source + Tmean_C:Water_source + Monthly_Q_mm:Water_source | HUC8_month_fe + year_fe | vcov: DK


###

Estimation:  OLS
Dep. var.: IWU_v1_mm, Fixed effects: HUC8_month_fe+year_fe
Inference:  DK
Observations:  81721

| Coefficient                              |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-----------------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Precip_mm:Water_source[Surface Water]    |     -0.632 |        0.010 |   -62.961 |      0.000 | -0.652 |  -0.612 |
| Precip_mm:Water_source[Groundwater]      |     -0.409 |        0.016 |   -25.365 |      0.000 | -0.441 |  -0.377 |
| Precip_mm:Water_source[Conjunctive]      |     -0.667 |        0.018 |   -37.479 |      0.000 | -0.703 |  -0.632 |
| Tmean_C:Water_source[Surface Water]      |      2.647 |        0.134 |    19.796 |      0.000 |  2.384 |   2.911 |
| Tmean_C:Water_source[Groundwater]        |      2.827 |        0.208 |    13.557 |      0.000 |  2.416 |   3.237 |
| Tmean_C:Water_source[Conjunctive]        |      2.191 |      

In [14]:
df_fe_mod_3 = save_panel_model_results(
    model=fe_mod_3,
    model_name='model_3',
    incorporate_water_source_info=False,
    panel_df=None,
    output_dir=PROJECT_ROOT / 'Data_main/Results',
    shapefile=None,
    spatial_unit_col='HUC8',
    rename_sp_unit='Water_source_class',
    shp_join_col=None,
    save_csv=True,
    save_shapefile=False
)
df_fe_mod_3

INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_3_results.csv


,Water_source_class,State,AQ_Region,P_2.5%,P_97.5%,P_Estimate,P_SE,P_SIG,P_p-value,P_t-stat,...,Q_SIG,Q_p-value,Q_t-stat,T_2.5%,T_97.5%,T_Estimate,T_SE,T_SIG,T_p-value,T_t-stat
0,Conjunctive,NaN,NaN,-0.702552,-0.632383,-0.667467,0.017809,1.0,0.0,-37.478504,...,0.0,0.467166,-0.728272,1.779744,2.602458,2.191101,0.208808,1.0,0.0,10.493368
1,Groundwater,NaN,NaN,-0.440671,-0.377152,-0.408912,0.016121,1.0,0.0,-25.364642,...,1.0,0.000000,10.095563,2.415858,3.237343,2.826601,0.208496,1.0,0.0,13.557089
2,Surface Water,NaN,NaN,-0.651843,-0.612288,-0.632065,0.010039,1.0,0.0,-62.960575,...,0.0,0.172316,1.368935,2.383847,2.910734,2.647290,0.133726,1.0,0.0,19.796363


## Model 4

`IWU_v1_mm ~ Precip_mm:AQ_Region + Tmean_C:AQ_Region + Monthly_Q_mm:AQ_Region | HUC8_month_fe + year_fe`

Middle ground between Model 2 (per HUC8, ~581 coefs) and Model 3 (per water-source class, 3 coefs). Estimates one P/T/Q sensitivity per AQ_Region (24 regions). Key goal: test whether GW-dominated regions are internally heterogeneous — e.g. arid SW basins (BR_AZ, BR_NV) vs. semi-arid HPA basins respond differently to precipitation. Only rows with a valid AQ_Region are used (~53% of the cleaned panel).

In [15]:
# filtering panel df.
# previously it was not filtered for 'AQ_Region column'ArithmeticError

pdf_m4 = pd.read_csv(PROJECT_ROOT / 'Data_main/panel_data/panel_data_monthly.csv')

# clean the panel df
# removing rows with NaN and selecting Irrigated HUC8s
pdf_m4 = clean_panel_df(pdf_m4, 
                     nan_cols_to_consider=['Precip_mm', 'Tmean_C', 'Monthly_Q_mm', 'AQ_Region'],
                     save_path=None,
                     years_to_consider=list(range(1986, 2020)))


# categorical column creation
categorical_config = {
    'HUC8': {
        'col_name' : 'HUC8',
        'assigned_categories' : list(pdf_m4.HUC8.unique()),
        'impose_order'    : False
    },
    
    'Water_source': {
        'col_name' : 'Water_source',
        'assigned_categories' : ['Surface Water', 'Groundwater', 'Conjunctive'],
        'impose_order'    : False
    },
    
    'AQ_Region': {
        'col_name' : 'AQ_Region',
        'assigned_categories' : list(pdf_m4.AQ_Region.unique()),
        'impose_order'    : False
    }
}

pdf_m4 = create_categorical_cols_in_df(pdf_m4, categorical_config)

# Fixed-effects column creation
FE_config = {
            'HUC8_month' : ['HUC8', 'month'],
            'year' : ['year']
        }

pdf_m4 = create_FE_columns_in_df(pdf_m4, FE_config,
                                  year_col='year', 
                                  month_col='month')


Step 1: Cleaned panel dataframe by dropping NaN rows and selecting irrigated HUC8s.
-------------------------------------------------------------------------------- 

HUC8s before cleaning the dataframe: 1187
HUC8s after cleaning the dataframe: 348

STEP 3: Categorical columns created: ['HUC8', 'Water_source', 'AQ_Region']
-------------------------------------------------------------------------------- 


STEP 4: Fixed effects columns created for: ['HUC8_month', 'year']
***** Not all FE columns will be included in the regression. *****
-------------------------------------------------------------------------------- 



In [16]:
fe_mod_4 = pyfixest_fit_FE(
    df=pdf_m4,
    target_col='IWU_v1_mm',
    regressor_cols=['Precip_mm', 'Tmean_C', 'Monthly_Q_mm'],
    fe_cols=['HUC8_month_fe', 
             'year_fe'
             ],
    include_base_regressors=False,
    interaction_dict={
        'Precip_mm'    : 'AQ_Region',
        'Tmean_C'      : 'AQ_Region',
        'Monthly_Q_mm' : 'AQ_Region',
    },
    add_linear_trend=False,
    unit_col='HUC8', trend_col='year',
    vcov_method='DK',
    vcov_col='time_id',
    bandwidth=24
)

fe_mod_4.summary()

INFO:panel_utils:Pyfixest model fitted. Formula: IWU_v1_mm ~ Precip_mm:AQ_Region + Tmean_C:AQ_Region + Monthly_Q_mm:AQ_Region | HUC8_month_fe + year_fe | vcov: DK


###

Estimation:  OLS
Dep. var.: IWU_v1_mm, Fixed effects: HUC8_month_fe+year_fe
Inference:  DK
Observations:  81721

| Coefficient                              |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-----------------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Precip_mm:AQ_Region[RG_NM]               |     -0.665 |        0.016 |   -42.521 |      0.000 | -0.696 |  -0.635 |
| Precip_mm:AQ_Region[BR_AZ_East]          |     -0.628 |        0.031 |   -20.230 |      0.000 | -0.689 |  -0.567 |
| Precip_mm:AQ_Region[UCRB_CO]             |     -0.645 |        0.025 |   -25.303 |      0.000 | -0.695 |  -0.595 |
| Precip_mm:AQ_Region[RG_CO]               |     -0.642 |        0.024 |   -27.125 |      0.000 | -0.689 |  -0.596 |
| Precip_mm:AQ_Region[HPA_TX_North]        |     -0.437 |        0.022 |   -19.810 |      0.000 | -0.481 |  -0.394 |
| Precip_mm:AQ_Region[BR_NV_Central]       |     -0.742 |      

In [17]:
df_fe_mod_4 = save_panel_model_results(
    model=fe_mod_4,
    model_name='model_4',
    incorporate_water_source_info=False,
    panel_df=None,
    output_dir=PROJECT_ROOT / 'Data_main/Results',
    shapefile=PROJECT_ROOT / 'Data_main/ref_shapes/aquifers_ROI/aquifers_by_state.shp',
    spatial_unit_col='AQ_Region',
    rename_sp_unit='AQ_Region',
    shp_join_col='AQ_Region',
    save_csv=True,
    save_shapefile=False
)
df_fe_mod_4.head()

INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_4_results.csv


,AQ_Region,State,AQ_Region,P_2.5%,P_97.5%,P_Estimate,P_SE,P_SIG,P_p-value,P_t-stat,...,Q_SIG,Q_p-value,Q_t-stat,T_2.5%,T_97.5%,T_Estimate,T_SE,T_SIG,T_p-value,T_t-stat
0,BR_AZ_East,Arizona,BR_AZ_East,-0.688887,-0.566623,-0.627755,0.031031,1.0,0.0,-20.229948,...,1.0,6.818015e-04,-3.442263,0.297768,2.683128,1.490448,0.605414,1.0,1.453483e-02,2.461865
1,BR_AZ_West,Arizona,BR_AZ_West,-0.756715,-0.548229,-0.652472,0.052915,1.0,0.0,-12.330689,...,1.0,3.935137e-02,-2.071957,-0.990928,2.012738,0.510905,0.762343,0.0,5.033967e-01,0.670178
2,BR_NV_Central,Nevada,BR_NV_Central,-0.790305,-0.694192,-0.742249,0.024394,1.0,0.0,-30.427742,...,1.0,7.145112e-07,5.094316,2.425994,3.082829,2.754412,0.166707,1.0,0.000000e+00,16.522423
3,BR_NV_North,Nevada,BR_NV_North,-0.864704,-0.773289,-0.818997,0.023201,1.0,0.0,-35.299465,...,1.0,1.735078e-03,3.168283,2.128781,3.207366,2.668073,0.273749,1.0,0.000000e+00,9.746414
4,BR_NV_West,Nevada,BR_NV_West,-0.842882,-0.682470,-0.762676,0.040713,1.0,0.0,-18.732932,...,1.0,1.242702e-05,4.464349,1.063999,2.338859,1.701429,0.323564,1.0,3.244345e-07,5.258394


----------
# IWU sensitivity shift across timeline
Here, the model structure is going to be the same as model 2, but we will split the timeline.

`IWU_v1_mm ~ Precip_anomaly:HUC8 + Tmean_anomaly:HUC8 + Monthly_Q_mm:HUC8 | HUC8_month_fe + year_fe`

In [18]:
# loading cleaned dataframe
cleaned_df = pd.read_csv(PROJECT_ROOT / 'Data_main/panel_data/panel_data_monthly_cleaned.csv')

# categorical column creation
categorical_config = {
    'HUC8': {
        'col_name' : 'HUC8',
        'assigned_categories' : list(cleaned_df.HUC8.unique()),
        'impose_order'    : False
        
    },
    
    'Water_source': {
        'col_name' : 'Water_source',
        'assigned_categories' : ['Surface Water', 'Groundwater', 'Conjunctive'],
        'impose_order'    : False
    }
}

cleaned_df = create_categorical_cols_in_df(cleaned_df, categorical_config)


# split dataframe for three timelines: 1986-1997, 1998-2008, 2009-2019
df_1st = cleaned_df[(cleaned_df['year'] >= 1986) & (cleaned_df['year'] <= 1997)].copy()
df_2nd = cleaned_df[(cleaned_df['year'] >= 1998) & (cleaned_df['year'] <= 2008)].copy()
df_3rd = cleaned_df[(cleaned_df['year'] >= 2009) & (cleaned_df['year'] <= 2019)].copy()


# Fixed-effects column creation
FE_config = {
            'HUC8_month' : ['HUC8', 'month'],
            'year' : ['year']
        }

df_1st = create_FE_columns_in_df(df_1st, FE_config, year_col='year', month_col='month')

df_2nd = create_FE_columns_in_df(df_2nd, FE_config, year_col='year', month_col='month')

df_3rd = create_FE_columns_in_df(df_3rd, FE_config, year_col='year', month_col='month')


# FE regression model

fed_timeline_1st = \
    pyfixest_fit_FE(df=df_1st, target_col='IWU_v1_mm', 
    regressor_cols=['Precip_mm', 'Tmean_C', 
                    'Monthly_Q_mm'], 
    fe_cols=['HUC8_month_fe', 'year_fe'],
    include_base_regressors=False, 
    interaction_dict={'Precip_mm': 'HUC8',
                      'Tmean_C': 'HUC8',
                      'Monthly_Q_mm': 'HUC8'},
    add_linear_trend=False,
    unit_col='HUC8', trend_col='year',
    vcov_method='DK',
    vcov_col='time_id', 
    bandwidth=12)

fed_timeline_2nd = \
    pyfixest_fit_FE(df=df_2nd, target_col='IWU_v1_mm', 
    regressor_cols=['Precip_mm', 'Tmean_C', 
                    'Monthly_Q_mm'], 
    fe_cols=['HUC8_month_fe', 'year_fe'],
    include_base_regressors=False, 
    interaction_dict={'Precip_mm': 'HUC8',
                     'Tmean_C': 'HUC8',
                     'Monthly_Q_mm': 'HUC8'},
    add_linear_trend=False,
    unit_col='HUC8', trend_col='year',
    vcov_method='DK',
    vcov_col='time_id', 
    bandwidth=12)

fed_timeline_3rd = \
    pyfixest_fit_FE(df=df_3rd, target_col='IWU_v1_mm', 
    regressor_cols=['Precip_mm', 'Tmean_C', 
                    'Monthly_Q_mm'], 
    fe_cols=['HUC8_month_fe', 'year_fe'],
    include_base_regressors=False, 
    interaction_dict={'Precip_mm': 'HUC8',
                        'Tmean_C': 'HUC8',
                        'Monthly_Q_mm': 'HUC8'},
    add_linear_trend=False,
    unit_col='HUC8', trend_col='year',
    vcov_method='DK',
    vcov_col='time_id', 
    bandwidth=12)


STEP 3: Categorical columns created: ['HUC8', 'Water_source']
-------------------------------------------------------------------------------- 


STEP 4: Fixed effects columns created for: ['HUC8_month', 'year']
***** Not all FE columns will be included in the regression. *****
-------------------------------------------------------------------------------- 


STEP 4: Fixed effects columns created for: ['HUC8_month', 'year']
***** Not all FE columns will be included in the regression. *****
-------------------------------------------------------------------------------- 


STEP 4: Fixed effects columns created for: ['HUC8_month', 'year']
***** Not all FE columns will be included in the regression. *****
-------------------------------------------------------------------------------- 



/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 267 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            124 variables dropped due to multicollinearity.
            The following variables are dropped: 
    Precip_mm:HUC8[11130102]
    Precip_mm:HUC8[11130105]
    Precip_mm:HUC8[12060101]
    Precip_mm:HUC8[12090103]
    Precip_mm:HUC8[12090105]
    ....
            
  warnings.warn(
INFO:panel_utils:Pyfixest model fitted. Formula: IWU_v1_mm ~ Precip_mm:HUC8 + Tmean_C:HUC8 + Monthly_Q_mm:HUC8 | HUC8_month_fe + year_fe | vcov: DK
/Users/mdfahimhasan/miniconda3/envs/westus_gw/lib/python3.10/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 5 singleton fixed effect(s) detected. These observation

-----------
*In the 1st timeline's dataframe, the 124 dropped variables correspond to HUC8s that have only a few observations in 1st dataframe (e.g., a HUC8 that only became irrigated in 1997). The HUC8_month_fe absorbs all remaining variation for these sparse units, making the interaction term Precip_mm:HUC8[X] perfectly collinear. Pyfixest drops them automatically.*

---

In [19]:
# save model results for all timelines
save_panel_model_results(
        model=fed_timeline_1st,
        model_name='model_timeline_1st',
        incorporate_water_source_info=True,
        panel_df=df_1st,
        output_dir=PROJECT_ROOT / 'Data_main/Results',
        shapefile=PROJECT_ROOT / 'Data_main/ref_shapes/WestUS_HUC8_processed.shp',
        spatial_unit_col='HUC8',
        rename_sp_unit='HUC8',
        shp_join_col='HUC8',
        save_csv=True,
        save_shapefile=False)

save_panel_model_results(
        model=fed_timeline_2nd,
        model_name='model_timeline_2nd',
        incorporate_water_source_info=True,
        panel_df=df_2nd,
        output_dir=PROJECT_ROOT / 'Data_main/Results',
        shapefile=PROJECT_ROOT / 'Data_main/ref_shapes/WestUS_HUC8_processed.shp',
        spatial_unit_col='HUC8',
        rename_sp_unit='HUC8',
        shp_join_col='HUC8',
        save_csv=True,
        save_shapefile=False)

save_panel_model_results(
        model=fed_timeline_3rd,
        model_name='model_timeline_3rd',
        incorporate_water_source_info=True,
        panel_df=df_3rd,
        output_dir=PROJECT_ROOT / 'Data_main/Results',
        shapefile=PROJECT_ROOT / 'Data_main/ref_shapes/WestUS_HUC8_processed.shp',
        spatial_unit_col='HUC8',
        rename_sp_unit='HUC8',
        shp_join_col='HUC8',
        save_csv=True,
        save_shapefile=False)

INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_timeline_1st_results.csv
INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_timeline_2nd_results.csv
INFO:panel_utils:Results CSV saved → /Users/mdfahimhasan/Documents/PROJECTS/WestUS_IWU_trend/Data_main/Results/model_timeline_3rd_results.csv


,HUC8,State,AQ_Region,Water_source,P_2.5%,P_97.5%,P_Estimate,P_SE,P_SIG,P_p-value,...,Q_SIG,Q_p-value,Q_t-stat,T_2.5%,T_97.5%,T_Estimate,T_SE,T_SIG,T_p-value,T_t-stat
0,10020001,Montana,NaN,NaN,-0.866527,-0.605227,-0.735877,0.065598,1.0,0.000000e+00,...,0.0,0.074259,1.809939,1.599427,3.351639,2.475533,0.439885,1.0,2.914240e-07,5.627687
1,10020002,Montana,NaN,NaN,-0.832735,-0.664128,-0.748432,0.042328,1.0,0.000000e+00,...,0.0,0.052649,1.968571,2.316880,3.380471,2.848676,0.267009,1.0,2.220446e-16,10.668823
2,10020003,Montana,NaN,NaN,-0.837543,-0.671355,-0.754449,0.041721,1.0,0.000000e+00,...,0.0,0.353240,0.934043,2.369442,3.710195,3.039818,0.336590,1.0,1.163514e-13,9.031232
3,10020004,Montana,NaN,NaN,-0.863500,-0.651670,-0.757585,0.053179,1.0,0.000000e+00,...,0.0,0.237550,1.190514,2.521195,4.384160,3.452677,0.467688,1.0,1.659362e-10,7.382430
4,10020005,Montana,NaN,NaN,-0.766586,-0.635829,-0.701207,0.032826,1.0,0.000000e+00,...,0.0,0.052447,1.970302,2.146522,3.119099,2.632810,0.244161,1.0,0.000000e+00,10.783092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,9020109,NaN,NaN,NaN,-0.682060,-0.442618,-0.562339,0.060111,1.0,2.797762e-14,...,0.0,0.201325,1.288949,0.296951,4.669345,2.483148,1.097669,1.0,2.654253e-02,2.262202
576,9020203,NaN,NaN,NaN,-0.523474,-0.340442,-0.431958,0.045949,1.0,2.287059e-14,...,0.0,0.193415,-1.312160,1.669423,3.313494,2.491459,0.412736,1.0,5.369828e-08,6.036445
577,9020204,NaN,NaN,NaN,-0.635046,-0.505346,-0.570196,0.032561,1.0,0.000000e+00,...,0.0,0.671518,0.425717,0.614489,3.186533,1.900511,0.645700,1.0,4.304359e-03,2.943336
578,9020307,NaN,NaN,NaN,-0.615965,-0.452721,-0.534343,0.040982,1.0,0.000000e+00,...,1.0,0.049763,1.993788,1.012137,3.392462,2.202299,0.597569,1.0,4.257830e-04,3.685428


In [22]:
# Load results
cols = ['HUC8', 'AQ_Region', 'Water_source', 'P_Estimate', 'P_SIG', 'T_Estimate', 'T_SIG']

df_1st_res = pd.read_csv(PROJECT_ROOT / 'Data_main/Results/model_timeline_1st_results.csv')[cols]
df_2nd_res = pd.read_csv(PROJECT_ROOT / 'Data_main/Results/model_timeline_2nd_results.csv')[cols]
df_3rd_res = pd.read_csv(PROJECT_ROOT / 'Data_main/Results/model_timeline_3rd_results.csv')[cols]

df_1st_rest = df_1st_res.rename(
    columns={'P_Estimate': 'P_Estimate_1st', 'P_SIG': 'P_SIG_1st',
             'T_Estimate': 'T_Estimate_1st', 'T_SIG': 'T_SIG_1st'})

df_2nd_rest = df_2nd_res.rename(
    columns={'P_Estimate': 'P_Estimate_2nd', 'P_SIG': 'P_SIG_2nd',
             'T_Estimate': 'T_Estimate_2nd', 'T_SIG': 'T_SIG_2nd'})

df_3rd_rest = df_3rd_res.rename(
    columns={'P_Estimate': 'P_Estimate_3rd', 'P_SIG': 'P_SIG_3rd',
             'T_Estimate': 'T_Estimate_3rd', 'T_SIG': 'T_SIG_3rd'})

# merge results
merged_res = df_1st_rest.merge(df_2nd_rest[['HUC8', 'P_Estimate_2nd', 
                                            'P_SIG_2nd', 'T_Estimate_2nd', 
                                            'T_SIG_2nd']], 
                               on='HUC8', how='left')

merged_res = merged_res.merge(df_3rd_rest[['HUC8', 'P_Estimate_3rd', 
                                           'P_SIG_3rd', 'T_Estimate_3rd', 
                                           'T_SIG_3rd']], 
                              on='HUC8', how='left')

# save results
merged_res.to_csv(PROJECT_ROOT / 'Data_main/Results/model_timeline_results_merged.csv', 
                  index=False)

merged_res.head()

,HUC8,AQ_Region,Water_source,P_Estimate_1st,P_SIG_1st,T_Estimate_1st,T_SIG_1st,P_Estimate_2nd,P_SIG_2nd,T_Estimate_2nd,T_SIG_2nd,P_Estimate_3rd,P_SIG_3rd,T_Estimate_3rd,T_SIG_3rd
0,10020001,NaN,NaN,-0.730610,1.0,2.246126,1.0,-0.704310,1.0,2.211515,1.0,-0.735877,1.0,2.475533,1.0
1,10020002,NaN,NaN,-0.728228,1.0,3.059949,1.0,-0.668795,1.0,3.008063,1.0,-0.748432,1.0,2.848676,1.0
2,10020003,NaN,NaN,-0.760103,1.0,3.624384,1.0,-0.693803,1.0,3.244134,1.0,-0.754449,1.0,3.039818,1.0
3,10020004,NaN,NaN,-0.829475,1.0,3.440043,1.0,-0.703063,1.0,4.121335,1.0,-0.757585,1.0,3.452677,1.0
4,10020005,NaN,NaN,-0.733251,1.0,2.275758,1.0,-0.637924,1.0,3.068515,1.0,-0.701207,1.0,2.632810,1.0
